# Cached PURE clips → augmentation → model → hybrid loss → backward pass → optimizer

In [2]:
from pathlib import Path
from types import SimpleNamespace
import gc
import os
import random
import sys
import time

import numpy as np
import scipy.__config__ as scipy_config
import torch
from torch.utils.data import DataLoader

# ============================================================
# PATHS AND SETTINGS
# ============================================================

PROJECT_ROOT = Path(
    "/media/data/rPPG/Code/GitHub/Catch_The_Mamba"
)

OFFICIAL_ROOT = PROJECT_ROOT / "official" / "RhythmMamba"

PURE_CONFIG = (
    PROJECT_ROOT
    / "configs"
    / "local"
    / "intra"
    / "PURE_RHYTHMMAMBA_LOCAL.yaml"
)

assert OFFICIAL_ROOT.exists()
assert PURE_CONFIG.exists()

if str(OFFICIAL_ROOT) not in sys.path:
    sys.path.insert(0, str(OFFICIAL_ROOT))


# ============================================================
# CLEAN PREVIOUS GPU OBJECTS
# ============================================================

for variable_name in [
    "model",
    "input_video",
    "prediction",
]:
    globals().pop(variable_name, None)

gc.collect()
torch.cuda.empty_cache()


# ============================================================
# REPRODUCIBILITY
# ============================================================

SEED = 100

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


# ============================================================
# COMPATIBILITY FIX FOR UNUSED MMPD IMPORT
# ============================================================

if not hasattr(scipy_config, "get_info"):
    scipy_config.get_info = lambda *args, **kwargs: {}


# ============================================================
# IMPORT OFFICIAL CODE
# ============================================================

from config import get_config
from dataset.data_loader.PURELoader import PURELoader
from neural_methods.trainer.RhythmMambaTrainer import RhythmMambaTrainer


# ============================================================
# LOAD CONFIGURATION AND DATASET
# ============================================================

config = get_config(
    SimpleNamespace(config_file=str(PURE_CONFIG))
)

assert config.TOOLBOX_MODE == "train_and_test"
assert Path(config.TRAIN.DATA.CACHED_PATH).exists()
assert Path(config.TRAIN.DATA.FILE_LIST_PATH).exists()

train_dataset = PURELoader(
    name="train",
    data_path=config.TRAIN.DATA.DATA_PATH,
    config_data=config.TRAIN.DATA,
)

train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=config.TRAIN.BATCH_SIZE,
    shuffle=True,
    num_workers=4,
)

data_loader_dictionary = {
    "train": train_loader,
    "valid": None,
    "test": None,
}


# ============================================================
# INITIALIZE OFFICIAL TRAINER
# ============================================================

trainer = RhythmMambaTrainer(
    config=config,
    data_loader=data_loader_dictionary,
)

batch = next(iter(train_loader))

data = batch[0].float()
labels = batch[1].float()

original_data_shape = tuple(data.shape)
original_label_shape = tuple(labels.shape)

batch_size = data.shape[0]


# ============================================================
# APPLY OFFICIAL TRAINING AUGMENTATION
# ============================================================

if config.TRAIN.AUG:
    data, labels = trainer.data_augmentation(
        data=data,
        labels=labels,
        index1=batch[2],
        index2=batch[3],
    )

assert data.shape == batch[0].shape
assert labels.shape == batch[1].shape
assert torch.isfinite(data).all()
assert torch.isfinite(labels).all()


# ============================================================
# ONE COMPLETE TRAINING STEP
# ============================================================

device = torch.device(config.DEVICE)

data = data.to(device)
labels = labels.to(device)

torch.cuda.reset_peak_memory_stats(device)
torch.cuda.synchronize(device)

start_time = time.perf_counter()

trainer.model.train()
trainer.optimizer.zero_grad()

predicted_ppg = trainer.model(data)

predicted_ppg = (
    predicted_ppg
    - torch.mean(
        predicted_ppg,
        dim=-1,
        keepdim=True,
    )
) / torch.std(
    predicted_ppg,
    dim=-1,
    keepdim=True,
)

loss = 0.0

for sample_index in range(batch_size):
    sample_loss = trainer.criterion(
        predicted_ppg[sample_index],
        labels[sample_index],
        0,
        config.TRAIN.DATA.FS,
        trainer.diff_flag,
    )

    loss = loss + sample_loss

loss = loss / batch_size

assert torch.isfinite(loss), (
    f"Training loss is not finite: {loss}"
)

loss.backward()

gradients_are_finite = all(
    torch.isfinite(parameter.grad).all()
    for parameter in trainer.model.parameters()
    if parameter.grad is not None
)

assert gradients_are_finite, (
    "At least one model gradient is NaN or infinite."
)

trainer.optimizer.step()
trainer.scheduler.step()

torch.cuda.synchronize(device)

elapsed_time = time.perf_counter() - start_time

peak_memory_gb = (
    torch.cuda.max_memory_allocated(device)
    / (1024**3)
)


# ============================================================
# RESULTS
# ============================================================

print("=" * 70)
print("OFFICIAL RHYTHMMAMBA TRAINING-STEP SMOKE TEST")
print("=" * 70)
print("Dataset clips       :", len(train_dataset))
print("Training batches    :", len(train_loader))
print("Configured batch    :", config.TRAIN.BATCH_SIZE)
print("Input shape         :", original_data_shape)
print("Label shape         :", original_label_shape)
print("Prediction shape    :", tuple(predicted_ppg.shape))
print("Augmentation enabled:", bool(config.TRAIN.AUG))
print("Loss                :", float(loss.detach().cpu()))
print("Gradients finite    :", gradients_are_finite)
print(f"Training-step time  : {elapsed_time:.3f} seconds")
print(f"Peak GPU memory     : {peak_memory_gb:.3f} GB")
print("GPU                 :", torch.cuda.get_device_name(device))

print("\nOfficial RhythmMamba training-step smoke test: PASSED")

=> Merging a config file from /media/data/rPPG/Code/GitHub/Catch_The_Mamba/configs/local/intra/PURE_RHYTHMMAMBA_LOCAL.yaml
Cached Data Path /media/data/rPPG/rPPG_Data/Mamba_Hunt/RhythmMamba_Preprocessed/PURE/PURE_SizeW128_SizeH128_ClipLength160_DataTypeStandardized_DataAugNone_LabelTypeStandardized_Crop_faceTrue_Large_boxTrue_Large_size1.5_Dyamic_DetFalse_det_len30_Median_face_boxFalse

File List Path /media/data/rPPG/rPPG_Data/Mamba_Hunt/RhythmMamba_Preprocessed/PURE/DataFileLists/PURE_SizeW128_SizeH128_ClipLength160_DataTypeStandardized_DataAugNone_LabelTypeStandardized_Crop_faceTrue_Large_boxTrue_Large_size1.5_Dyamic_DetFalse_det_len30_Median_face_boxFalse_0.0_0.6.csv
 train Preprocessed Dataset Length: 443



/home/rafsan/miniconda3/envs/mamba_hunting/lib/python3.11/site-packages/torch/cuda/__init__.py:611: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")
/media/data/rPPG/Code/GitHub/Catch_The_Mamba/official/RhythmMamba/neural_methods/model/RhythmMamba.py:136: UserWarning: Casting complex values to real discards the imaginary part (Triggered internally at ../aten/src/ATen/native/Copy.cpp:299.)
  x = x.to(torch.float32)
/home/rafsan/miniconda3/envs/mamba_hunting/lib/python3.11/site-packages/torch/nn/_reduction.py:42: UserWarning: size_average and reduce args will be deprecated, please use reduction='none' instead.
  warnings.warn(warning.format(ret))


OFFICIAL RHYTHMMAMBA TRAINING-STEP SMOKE TEST
Dataset clips       : 443
Training batches    : 28
Configured batch    : 16
Input shape         : (16, 160, 3, 128, 128)
Label shape         : (16, 160)
Prediction shape    : (16, 160)
Augmentation enabled: True
Loss                : 4.851820945739746
Gradients finite    : True
Training-step time  : 22.207 seconds
Peak GPU memory     : 9.105 GB
GPU                 : Tesla V100-PCIE-32GB

Official RhythmMamba training-step smoke test: PASSED
